[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/templates/38_grpo_loss.ipynb)

# 🔴 Hard: GRPO (Group Relative Policy Optimization) Loss

*RLHF & Preference Losses*
Implement the **GRPO** loss — the objective behind DeepSeek-R1-style reasoning
training.

**1. Group-relative advantages.** For a prompt with a group of $G$ sampled
completions and rewards $r_1 \dots r_G$:

$$A_i = \frac{r_i - \text{mean}(r)}{\text{std}(r) + \epsilon}$$

Every token of completion $i$ gets that same scalar advantage.

**2. Clipped surrogate with a KL penalty:**

$$\mathcal{L} = -\frac{1}{\sum m}\sum m \Big[
\min\big(r_t A,\ \text{clip}(r_t, 1-\epsilon_c, 1+\epsilon_c) A\big)
- \beta\, \mathbb{D}_{KL}\big[\pi_\theta \,\|\, \pi_{\text{ref}}\big]_t \Big]$$

**3. The KL term** uses the low-variance, always-positive **k3** estimator:

$$\mathbb{D}_{KL} = \exp(\ell_{\text{ref}} - \ell_\theta) - (\ell_{\text{ref}} - \ell_\theta) - 1$$

### Signature
```python
def grpo_loss(new_logps, old_logps, ref_logps, rewards,
              clip_eps=0.2, beta=0.04, mask=None):
    # new_logps / old_logps / ref_logps: (n_prompts, group_size, seq)
    # rewards:                           (n_prompts, group_size)
    # mask:                              (n_prompts, group_size, seq) or None
    ...  # -> scalar loss
```

### Rules
- Normalise rewards **within each group**, never across the whole batch — that
  is the entire idea
- Broadcast the per-sequence advantage across the token axis
- Use the k3 KL estimator above, not `logp_ratio` and not a plain difference
- Mask-average over real tokens
- Do not use any RL library

### Why the group baseline replaces the value network
PPO needs an advantage, and an advantage needs a baseline $V(s)$ — normally a
second network of the same size as the policy, trained alongside it. That is
expensive, and for LLM reasoning it is also *hard*: the value of a
half-finished chain of thought is a terrible regression target.

GRPO's move is to sample $G$ completions for the **same** prompt and use the
group's own mean reward as the baseline. Same prompt means the comparison is
apples-to-apples, the variance reduction is large, and it costs no parameters.
Be precise about the "unbiased" claim in an interview, though: a baseline is
unbiased when it is *independent of the sampled action*, and the group mean
includes completion $i$ itself, so it introduces an $O(1/G)$ bias that a
leave-one-out mean would not.

Dividing by the group std whitens the advantage scale, which is what lets one
learning rate work across prompts of wildly different difficulty — but it is
also the most-questioned line in the method. It up-weights groups that happen to
have low reward variance, i.e. the prompts that are nearly always right or
nearly always wrong, which is precisely the wrong place to put gradient. Several
2025 variants drop the division and keep only the mean-centering. Implement it
as specified here, and know why someone might not.

### The degenerate case to watch
If every completion in a group gets the **same** reward — all correct, or all
wrong — then $\text{std} = 0$ and every advantage is $0$. That group
contributes no policy gradient at all, only the KL term. This is not a bug; it
is why GRPO implementations care about prompts landing at neither 0% nor 100%
pass rate.

### Why k3 and not the naive estimator
The naive single-sample KL, $\ell_\theta - \ell_{\text{ref}}$, is unbiased but
can go **negative** for an individual sample, which makes a "penalty" that
sometimes pays you. The k3 form $e^{-x} - (-x) - 1$ with
$x = \ell_\theta - \ell_{\text{ref}}$ is non-negative everywhere, is zero
exactly when the policies agree, and has much lower variance.

In [ ]:
# Install jax-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q jax-judge flax')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

import jax
import jax.numpy as jnp


def grpo_loss(new_logps, old_logps, ref_logps, rewards,
              clip_eps=0.2, beta=0.04, mask=None):
    """GRPO loss.

    Args:
        new_logps: (n_prompts, group_size, seq) current policy log-probs
        old_logps: (n_prompts, group_size, seq) sampling policy log-probs
        ref_logps: (n_prompts, group_size, seq) reference policy log-probs
        rewards:   (n_prompts, group_size) scalar reward per completion
        clip_eps:  PPO clip range
        beta:      KL penalty weight
        mask:      optional (n_prompts, group_size, seq), 1 real / 0 padding

    Returns:
        Scalar loss.
    """
    pass  # Replace this

In [ ]:
# 🔍 Scratch cell — poke at your implementation
import jax.numpy as jnp

# One prompt, four completions, rewards 0/0/1/1.
rewards = jnp.array([[0.0, 0.0, 1.0, 1.0]])
mean = rewards.mean(-1, keepdims=True)
std = rewards.std(-1, keepdims=True)
print("advantages:", ((rewards - mean) / (std + 1e-8))[0])

lp = jnp.zeros((1, 4, 3))
print("loss (on-policy, no KL):", float(grpo_loss(lp, lp, lp, rewards, beta=0.0)))

# All-correct group -> std 0 -> every advantage 0 -> no policy gradient.
flat = jnp.ones((1, 4))
print("loss (all rewards equal):", float(grpo_loss(lp, lp, lp, flat, beta=0.0)))

In [ ]:
# ✅ SUBMIT — run this cell to check your solution
from jax_judge import check, hint, solution

check("grpo_loss")

# hint("grpo_loss")      # stuck? nudge without the answer
# solution("grpo_loss")  # spoiler: the reference implementation